In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv
/kaggle/input/datasets/kalyandath/faiss-index/faiss_index/index.faiss
/kaggle/input/datasets/kalyandath/faiss-index/faiss_index/index.pkl


In [2]:
!pip install -q langchain_huggingface
!pip install -q langchain_community
!pip install -q faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 59.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
moviepy 1.0.3 require

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-miniLM-L6-v2",
    model_kwargs = {"device":"cuda"}
)

FAISS_INDEX_PATH = "/kaggle/input/datasets/kalyandath/faiss-index/faiss_index"

vector_store = FAISS.load_local(
    folder_path = FAISS_INDEX_PATH,
    embeddings = embeddings,
    allow_dangerous_deserialization = True
)

/tmp/ipykernel_59/2735913336.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [65]:
query = "Identify the correct statement: What is the concept of simultaneity in Einstein's book, Relativity? carefully."

docs_scores = vector_store.similarity_search_with_score(query, k=3)

for doc,score in docs_scores:
    print(f"Score(distance): {score:.4f}")
    print(f"Source: {doc.metadata['source']}")
    print(f"Content: {doc.page_content}....\n")

Score(distance): 0.8999
Source: Physical definition
Content: . He showed that this postulate, together with a reasonable definition for what it means for two events to be simultaneous, requires that distances appear compressed and time intervals appear lengthened for events associated with objects in motion relative to an inertial observer.
The theory of special relativity finds a convenient formulation in Minkowski spacetime, a mathematical structure that combines three dimensions of space with a single dimension of time....

Score(distance): 0.9014
Source: The metric and four-vectors
Content: .  In special relativity....

Score(distance): 0.9436
Source: History
Content: . [...] He told me later that it came to him as a great shock when Einstein published his paper in which the equivalence of the different local times of observers moving relative to each other was pronounced; for he had reached the same conclusions independently but did not publish them because he wished first to work

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret = user_secrets.get_secret("HF_TOKEN")


In [5]:
MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL, token = secret)

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Quantization

In [6]:
!pip install -q bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.3 MB/s eta 0:00:00:00:0100:01


In [7]:
from transformers import BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_compute_dtype = torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config = bnb_config,
    device_map = "auto",
    token = secret
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

In [173]:
text_generation_pipeline = pipeline(
    "text-generation",
    model = model,
    tokenizer = tokenizer,
    max_new_tokens = 128,
    temperature = 0.1,
    repetition_penalty = 1.1,
    return_full_text = False
)

llm = HuggingFacePipeline(pipeline = text_generation_pipeline)

In [174]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

prompt_template='''<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an expert scientific multiple-choice answering assistant.
Your task is to answer the question using ONLY the provided context.
Do NOT write an analysis. Do NOT explain your reasoning.
You MUST output ONLY your top 3 chosen letters wrapped in XML tags.
Example format: <answer>C A E</answer>
<|eot_id|><|start_header_id|>user<|end_header_id|>

Context:
{context}

Question and options: 
{question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>'''

PROMPT = PromptTemplate(
    template = prompt_template, input_variables = ["context", "question"]
)

qa_chain = RetrievalQA.from_chain_type(
    llm = llm,
    chain_type = "stuff",
    retriever = vector_store.as_retriever(search_kwargs={"k":3}),
    return_source_documents = False,
    chain_type_kwargs={"prompt": PROMPT}
)

## Test the Rag

In [175]:
import re
def parse_answer_option(response):
    match = re.search(r'<answer>\s*(.*?)\s*</answer>', response, re.IGNORECASE)

    extracted_options = []

    if match:
        tag_content = match.group(1).upper()
        extracted_options = re.findall(r'[A-E]', tag_content)
    else:
        all_letters = re.findall(r'\b([A-E])\b', response)
        extracted_options = all_letters[-3:] if all_letters else []

    top_3 = list(dict.fromkeys(extracted_options))
    top_3 = list(set(top_3))
    default_options = ['A', 'B', 'C', 'D', 'E']
    for letter in default_options:
        if len(top_3) >= 3:
            break
        if letter not in top_3:
            top_3.append(letter)

    
    print(top_3)

    return " ".join(top_3[:3])
    

In [176]:
parse_answer_option("<answer>/answer> Dasfdsadf.safasdfasdf")

['A', 'B', 'C']


'A B C'

In [177]:
def query_rag(question):
    response = qa_chain.invoke({"query": question})
    # for doc in response['source_documents']:
    #     print(f"= {doc.metadata["source"]}")
    print("-" * 50)
    return response["result"]

In [178]:
optionA = 'A) Simultaneity is relative, meaning that two events that appear simultaneous to an observer in a particular inertial reference frame need not be judged as simultaneous by a second observer in a different inertial frame of reference.'
optionB = 'B) Simultaneity is relative, meaning that two events that appear simultaneous to an observer in a particular inertial reference frame will always be judged as simultaneous by a second observer in a different inertial frame of reference.'
optionC = 'C) Simultaneity is absolute, meaning that two events that appear simultaneous to an observer in a particular inertial reference frame will always be judged as simultaneous by a second observer in a different inertial frame of reference.'
optionD = 'D) Simultaneity is a concept that applies only to Newtonian theories and not to relativistic theories.'
optionE = 'E) Simultaneity is a concept that applies only to relativistic theories and not to Newtonian theories.'
options = optionA + "\n" + optionB + "\n" + optionC + "\n" + optionD + "\n" + optionE
query_rag(f"Identify the correct statement: What is the concept of simultaneity in Einstein's book, Relativity? carefully.\n\n{options}")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--------------------------------------------------


'\n\n<answer>A C</answer>'

In [71]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

In [179]:
import random
max_length = len(train_df)

index = random.randint(0,max_length)

prompt = str(train_df.iloc[index]["prompt"])
options = ""
for item in ["A", "B", "C", "D", "E"]:
    options += f"{item}) {str(train_df.iloc[index][item])}\n"
print(prompt)
print(options)
print(f"Answer(Truth Value): {train_df.iloc[index]["answer"]}\n\n")

query = prompt + "\n\n" + options
print(query_rag(query))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Pick the best possible answer: What is the propagation constant in sinusoidal waves?
A) The propagation constant is a measure of the amplitude of the sinusoidal wave that varies with distance.
B) The propagation constant is a real number that remains constant with distance due to the phase change in the sinusoidal wave.
C) The propagation constant is a real number that varies with distance due to the phase change in the sinusoidal wave.
D) The propagation constant is a complex number that varies with distance due to the phase change in the sinusoidal wave.
E) The propagation constant is a complex number that remains constant with distance due to the phase change in the sinusoidal wave.

Answer(Truth Value): D


--------------------------------------------------


<answer>D C</answer>


Try on test.csv

In [103]:
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [180]:
from tqdm.notebook import tqdm

predictions = []
for index, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Scoring Questions"):
    question = row["prompt"]

    options_text = (
        f"A. {row['A']}\n"
        f"B. {row['B']}\n"
        f"C. {row['C']}\n"
        f"D. {row['D']}\n"
        f"E. {row['E']}"
    )
   
    top3_prediction = parse_answer_option(query_rag(question + "\n\n" + options_text))

    predictions.append({
        "id": row["id"],
        "prediction": top3_prediction
    })


submission_df = pd.DataFrame(predictions)


Scoring Questions:   0%|          | 0/500 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--------------------------------------------------
['D', 'E', 'A']


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--------------------------------------------------
['D', 'B', 'C']


...

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--------------------------------------------------
['B', 'C', 'A']


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--------------------------------------------------
['D', 'B', 'C']
--------------------------------------------------
['C', 'A', 'B']


NameError: name 'output_csv_path' is not defined

In [181]:
submission_df.to_csv("submission.csv", index=False)